In [2]:
"""Enhanced Grazioso Salvare dashboard for CS 499 Database Enhancement.

This notebook interface uses the enhanced AnimalShelter database module.
For local demonstration, it loads controlled sample records into an
in-memory MongoDB-style collection through mongomock. The enhanced
AnimalShelter module remains capable of using a securely configured
MongoDB connection through environment variables.
"""

from typing import Any

import dash_leaflet as dl
import mongomock
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, dash_table, dcc, html

from animal_shelter import AnimalShelter


# ---------------------------------------------------------------------
# Controlled local demonstration data
# ---------------------------------------------------------------------

DEMO_RECORDS = [
    {
        "animal_id": "A100",
        "name": "Marina",
        "animal_type": "Dog",
        "breed": "Labrador Retriever Mix",
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": 52,
        "location_lat": 30.2672,
        "location_long": -97.7431,
    },
    {
        "animal_id": "A101",
        "name": "Harbor",
        "animal_type": "Dog",
        "breed": "Chesapeake Bay Retriever Mix",
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": 70,
        "location_lat": 30.2810,
        "location_long": -97.7520,
    },
    {
        "animal_id": "A102",
        "name": "River",
        "animal_type": "Dog",
        "breed": "Newfoundland Mix",
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": 88,
        "location_lat": 30.2550,
        "location_long": -97.7210,
    },
    {
        "animal_id": "A103",
        "name": "Ranger",
        "animal_type": "Dog",
        "breed": "German Shepherd",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": 64,
        "location_lat": 30.2920,
        "location_long": -97.7350,
    },
    {
        "animal_id": "A104",
        "name": "Summit",
        "animal_type": "Dog",
        "breed": "Siberian Husky",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": 78,
        "location_lat": 30.2610,
        "location_long": -97.7740,
    },
    {
        "animal_id": "A105",
        "name": "Tracker",
        "animal_type": "Dog",
        "breed": "Bloodhound",
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": 100,
        "location_lat": 30.2430,
        "location_long": -97.7560,
    },
    {
        "animal_id": "A106",
        "name": "Atlas",
        "animal_type": "Dog",
        "breed": "Rottweiler",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": 90,
        "location_lat": 30.3070,
        "location_long": -97.7460,
    },
    {
        "animal_id": "A107",
        "name": "Mittens",
        "animal_type": "Cat",
        "breed": "Domestic Shorthair Mix",
        "sex_upon_outcome": "Spayed Female",
        "age_upon_outcome_in_weeks": 40,
        "location_lat": 30.2750,
        "location_long": -97.7100,
    },
]


def create_demo_database() -> AnimalShelter:
    """Create a temporary database collection for local dashboard demonstration."""

    demo_client = mongomock.MongoClient()

    shelter = AnimalShelter.__new__(AnimalShelter)
    shelter.client = demo_client
    shelter.database = demo_client["aac"]
    shelter.collection = shelter.database["animals"]

    shelter.collection.insert_many(DEMO_RECORDS)

    return shelter


# Use a temporary MongoDB-style collection for the locally runnable dashboard.
shelter = create_demo_database()


# ---------------------------------------------------------------------
# Data preparation and visualization functions
# ---------------------------------------------------------------------

def records_to_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    """Convert database records into display-ready table data."""

    dataframe = pd.DataFrame(records)

    if "_id" in dataframe.columns:
        dataframe = dataframe.drop(columns=["_id"])

    return dataframe


def build_breed_chart(rescue_type: str):
    """Build a breed summary chart using the database aggregation method."""

    summary_records = shelter.breed_summary(rescue_type)
    summary_dataframe = pd.DataFrame(summary_records)

    if summary_dataframe.empty:
        figure = px.bar(title="No Matching Rescue Candidates")
        figure.update_layout(
            xaxis={"visible": False},
            yaxis={"visible": False},
            annotations=[
                {
                    "text": "No animals match the selected rescue specialization.",
                    "xref": "paper",
                    "yref": "paper",
                    "showarrow": False,
                    "font": {"size": 15},
                }
            ],
        )
        return figure

    summary_dataframe = summary_dataframe.rename(columns={"_id": "Breed", "count": "Candidates"})

    figure = px.bar(
        summary_dataframe,
        x="Breed",
        y="Candidates",
        title="Matching Rescue Candidates by Breed",
        labels={"Breed": "Breed", "Candidates": "Number of Candidates"},
    )

    figure.update_layout(
        margin={"l": 40, "r": 20, "t": 55, "b": 110},
        xaxis_tickangle=-35,
    )

    return figure


def build_location_map(
    records: list[dict[str, Any]],
    selected_rows: list[int] | None,
) -> dl.Map:
    """Build a map displaying the location of the selected animal record."""

    default_location = [30.2672, -97.7431]

    if not records:
        return dl.Map(
            center=default_location,
            zoom=11,
            style={"width": "100%", "height": "430px"},
            children=[dl.TileLayer()],
        )

    selected_index = selected_rows[0] if selected_rows else 0

    if selected_index >= len(records):
        selected_index = 0

    animal = records[selected_index]
    location = [animal["location_lat"], animal["location_long"]]

    return dl.Map(
        center=location,
        zoom=12,
        style={"width": "100%", "height": "430px"},
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=location,
                children=[
                    dl.Tooltip(f'{animal["name"]}: {animal["breed"]}'),
                    dl.Popup(
                        [
                            html.H4(animal["name"]),
                            html.P(f'Animal ID: {animal["animal_id"]}'),
                            html.P(f'Breed: {animal["breed"]}'),
                            html.P(f'Sex Upon Outcome: {animal["sex_upon_outcome"]}'),
                        ]
                    ),
                ],
            ),
        ],
    )


# ---------------------------------------------------------------------
# Initial dashboard data
# ---------------------------------------------------------------------

initial_records = shelter.find_rescue_candidates("All")
initial_dataframe = records_to_dataframe(initial_records)


# ---------------------------------------------------------------------
# Dashboard layout
# ---------------------------------------------------------------------

app = Dash(__name__)

app.layout = html.Div(
    [
        html.Div(
            [
                html.H1(
                    "Grazioso Salvare Rescue Candidate Dashboard",
                    style={"marginBottom": "8px"},
                ),
                html.P(
                    "Enhanced Database Artifact for CS 499",
                    style={"fontSize": "18px", "marginTop": "0"},
                ),
                html.P(
                    "This dashboard identifies animals suited for rescue training specializations using secure and validated database query logic.",
                    style={"fontSize": "14px"},
                ),
            ],
            style={
                "textAlign": "center",
                "padding": "22px",
                "backgroundColor": "#eef3f6",
                "borderRadius": "10px",
                "marginBottom": "20px",
            },
        ),
        html.Div(
            [
                html.Label(
                    "Select a rescue specialization:",
                    style={"fontWeight": "bold", "fontSize": "16px"},
                ),
                dcc.Dropdown(
                    id="rescue-type",
                    options=[
                        {"label": "All Animals", "value": "All"},
                        {"label": "Water Rescue", "value": "Water Rescue"},
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "Mountain or Wilderness Rescue",
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "Disaster or Individual Tracking",
                        },
                    ],
                    value="All",
                    clearable=False,
                    style={"maxWidth": "540px", "marginTop": "8px"},
                ),
                html.P(
                    id="candidate-count",
                    style={
                        "fontWeight": "bold",
                        "marginTop": "15px",
                        "fontSize": "15px",
                    },
                ),
            ],
            style={"marginBottom": "20px"},
        ),
        html.H3("Candidate Records"),
        dash_table.DataTable(
            id="candidate-table",
            columns=[
                {"name": column.replace("_", " ").title(), "id": column}
                for column in initial_dataframe.columns
            ],
            data=initial_dataframe.to_dict("records"),
            sort_action="native",
            filter_action="native",
            page_action="native",
            page_size=8,
            row_selectable="single",
            selected_rows=[0],
            style_table={"overflowX": "auto"},
            style_cell={
                "textAlign": "left",
                "padding": "9px",
                "fontFamily": "Arial",
                "fontSize": "13px",
                "minWidth": "110px",
            },
            style_header={
                "fontWeight": "bold",
                "backgroundColor": "#e6edf2",
                "border": "1px solid #c8d2d9",
            },
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.H3("Selected Animal Location"),
                        html.Div(id="candidate-map"),
                    ],
                    style={
                        "width": "48%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                    },
                ),
                html.Div(
                    [
                        html.H3("Breed Summary"),
                        dcc.Graph(id="breed-chart"),
                    ],
                    style={
                        "width": "48%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                        "marginLeft": "3%",
                    },
                ),
            ],
            style={"paddingTop": "30px"},
        ),
    ],
    style={
        "fontFamily": "Arial",
        "maxWidth": "1250px",
        "margin": "0 auto",
        "padding": "24px",
    },
)


# ---------------------------------------------------------------------
# Dashboard callbacks
# ---------------------------------------------------------------------

@app.callback(
    Output("candidate-table", "data"),
    Output("candidate-table", "selected_rows"),
    Output("candidate-count", "children"),
    Output("breed-chart", "figure"),
    Input("rescue-type", "value"),
)
def update_candidate_results(
    rescue_type: str,
) -> tuple[list[dict[str, Any]], list[int], str, Any]:
    """Update the table, count message, and breed chart after filtering."""

    records = shelter.find_rescue_candidates(rescue_type)
    dataframe = records_to_dataframe(records)

    specialization_name = (
        "all available animal records"
        if rescue_type == "All"
        else rescue_type.lower() + " candidates"
    )

    message = (
        f"Database results: {len(dataframe)} matching record(s) displayed for "
        f"{specialization_name}."
    )

    selected_rows = [0] if not dataframe.empty else []

    return (
        dataframe.to_dict("records"),
        selected_rows,
        message,
        build_breed_chart(rescue_type),
    )


@app.callback(
    Output("candidate-map", "children"),
    Input("candidate-table", "derived_virtual_data"),
    Input("candidate-table", "derived_virtual_selected_rows"),
)
def update_selected_animal_map(
    visible_records: list[dict[str, Any]] | None,
    selected_rows: list[int] | None,
) -> dl.Map:
    """Update the location map based on the selected displayed record."""

    records = visible_records if visible_records is not None else initial_records

    return build_location_map(records, selected_rows)


# ---------------------------------------------------------------------
# Launch the enhanced dashboard
# ---------------------------------------------------------------------

app.run(jupyter_mode="external", debug=True)

Dash app running on http://127.0.0.1:8050/
